In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utilities



In [0]:
print(bronze_schema,silver_schema,gold_schema)

**Bronze Layer Processing**

In [0]:
df_bronze = spark.read\
    .option("header",True)\
    .option("inferSchema",True)\
    .csv("s3://sportsbar-dp-sh/products/*.csv")
df_bronze=(
    df_bronze.withColumn("file_name",F.col("_metadata.file_name"))
    .withColumn("ingest_time",F.current_timestamp())
    .withColumn("file_size",F.col("_metadata.file_size")))

dbutils.widgets.text("catalog","fmcg","catalog")
dbutils.widgets.text("data_source","products","data_source")


catalog=dbutils.widgets.get("catalog")
data_source=dbutils.widgets.get("data_source")


df_bronze.write.format("delta")\
    .mode("overwrite")\
    .option("header",True)\
    .option("delta.enableChangeDataFeed",True)\
    .option("mergeSchema",True)\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

**Sliver Layer Processing**

In [0]:
df_silver=spark.table(f"{catalog}.{bronze_schema}.{data_source}")

print("Count of table before:",df_silver.count())

df_silver=df_silver.dropDuplicates(["product_id"])

print ("Count of table after:",df_silver.count())

df_silver=(df_silver.withColumn(
    "product",F.trim(F.split(F.col("product_name"),r"\(").getItem(0)))
           .withColumn(
    "variant",
    F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1))
)
df_silver=df_silver.withColumn("category",F.initcap(F.col("category")))
df_silver=df_silver.withColumn(
    "category",F.trim(F.regexp_replace(F.col("category"),"protien","protein"))
    )
df_silver=df_silver.withColumn(
    "division",F.when(F.col("category")=="Energy Bars","Nutrition Bars")
    .when(F.col("category")=="Protien Bars","Nutrition Bars")
    .when(F.col("category")=="Granola & Cereals","Breakfast Foods")
    .when(F.col("category")=="Recovery Dairy", "Dairy & Recovery")
    .when(F.col("category")=="Healthy Snacks","Healthy Snacks")
    .when(F.col("category")=="Electrolyte Mix","Hydration & Electrolytes")
    .otherwise("Other")
)

df_silver=df_silver.withColumn("product_code",F.sha2(F.col("product").cast("string"),256))
df_silver=df_silver.withColumn(
        "product_id",
        F.when(
            F.col("product_id").cast("string").rlike("^[0-9]"),
            F.col("product_id").cast("string")
        ).otherwise(F.lit(999999).cast("string"))
    )
df_silver = df_silver.select("product_code", "division", "category", "product", "variant", "product_id", "ingest_time", "file_name", "file_size")

df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("header",True)\
    .option("delta.enableChangeDataFeed",True)\
    .option("mergeSchema",True)\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

**Gold Layer Processing **

In [0]:
df_gold=spark.table(f"{catalog}.{silver_schema}.{data_source}")

df_gold=df_gold.select("product_code","division","category","product","variant")

df_gold.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema",True)\
    .option("header",True)\
    .option("delta.enableChangeDataFeed",True)\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
delta_table=DeltaTable.forName(spark,"fmcg.gold.dim_products")
df_child_customer=spark.table("fmcg.gold.sb_dim_products")

delta_table.alias("target").merge(
    df_child_customer.alias("source"),
    "target.product_code = source.product_code and target.division=source.division and target.category=source.category and target.product=source.product and target.variant=source.variant"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()